# 03. @jit と vmap — 高速化とベクトル化

**このノートブックの内容**:
- `@jax.jit`: 関数を JIT コンパイルで高速化
- `jax.vmap`: for ループの代わりに「自動ベクトル化」
- ベンチマーク比較

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (03_jit_vmap.md)](../03_jit_vmap.md)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)

print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
%matplotlib inline

## 1. `@jax.jit` で関数を高速化

JAX は関数を XLA (Accelerated Linear Algebra) にコンパイルして実行可能。
初回はコンパイル時間が発生するが、2 回目以降は爆速。

In [ ]:
import time

def compute(x):
    for _ in range(50):
        x = jnp.sin(x) * jnp.cos(x) + jnp.exp(-x**2)
    return x

compute_jit = jax.jit(compute)

x = jnp.linspace(-3, 3, 100_000)

# JIT 無し
t0 = time.perf_counter()
_ = compute(x).block_until_ready()
t_plain = time.perf_counter() - t0

# JIT 有り (warm up + 計測)
_ = compute_jit(x).block_until_ready()  # コンパイル
t0 = time.perf_counter()
_ = compute_jit(x).block_until_ready()
t_jit = time.perf_counter() - t0

print(f'JIT 無し: {t_plain * 1000:.2f} ms')
print(f'JIT 有り: {t_jit * 1000:.2f} ms  ({t_plain / t_jit:.1f}x 高速化)')

## 2. `jax.vmap` — for ループの代わりに 1 関数で何回も適用

`vmap(f)` は「f を配列の各要素に適用」 を自動でやってくれる。
for ループ + JIT より高速かつ簡潔。

In [ ]:
# 単一サンプル用の関数
def predict(w, x):
    '''単一の入力 x に対する予測 (例: 線形モデル).'''
    return jnp.dot(w, x)

# バッチに対して predict を vmap で適用
batched_predict = jax.vmap(predict, in_axes=(None, 0))
# in_axes=(None, 0) は「w はバッチ軸なし、x の 0 番目軸がバッチ」

w = jnp.array([1.0, 2.0, 3.0])
X_batch = jnp.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 1],
])  # 4 サンプル × 3 特徴

results = batched_predict(w, X_batch)
print(f'バッチ予測結果: {results}')
print('  ↑ 各行は w · x_i を計算した結果')

## 3. `vmap` + `grad` — バッチ全体の勾配を 1 行で

ML の典型: 「サンプルごとの損失の勾配をバッチ全体で計算」。

In [ ]:
def loss_per_sample(w, x, y):
    pred = jnp.dot(w, x)
    return (pred - y) ** 2

# サンプルごとの勾配関数
grad_per_sample = jax.grad(loss_per_sample, argnums=0)

# バッチに対して vmap で適用
batched_grad = jax.vmap(grad_per_sample, in_axes=(None, 0, 0))

w = jnp.array([0.5, 0.5, 0.5])
X = jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=jnp.float32)
y = jnp.array([1.0, 2.0, 3.0])

grads = batched_grad(w, X, y)
print(f'各サンプルの勾配:\n{grads}')
print(f'\n平均勾配 (∇L バッチ平均): {grads.mean(axis=0)}')

## 4. `jit` + `vmap` + `grad` の組合せ — JAX らしい書き方

3 つの変換 (transformation) を合成: 「勾配計算 → バッチ化 → コンパイル」 を一気に。

In [ ]:
@jax.jit
def train_step(w, X_batch, y_batch, lr=0.01):
    '''バッチ全体での 1 ステップ学習.'''
    # バッチ勾配の平均
    def per_sample_loss(w, x, y):
        return (jnp.dot(w, x) - y) ** 2
    grads = jax.vmap(jax.grad(per_sample_loss), in_axes=(None, 0, 0))(w, X_batch, y_batch)
    mean_grad = grads.mean(axis=0)
    return w - lr * mean_grad

# 訓練ループ
key = jax.random.PRNGKey(0)
w = jax.random.normal(key, (3,)) * 0.1
X = jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 1]], dtype=jnp.float32)
y = jnp.array([1.0, 2.0, 3.0, 6.0])  # 真の w = (1, 2, 3) に対する正解

for step in range(100):
    w = train_step(w, X, y)

print(f'最終 w = {w}  (真の w = [1, 2, 3])')
print('→ JAX の transformation 合成で簡潔に学習が書ける')

## 5. なぜ vmap が速いのか — 内部で SIMD/並列化される

素朴な for ループだと逐次実行になるが、`vmap` は内部でバッチ次元を意識した最適化が入る。
GPU/TPU では更に並列度が上がる。

**まとめ: JAX らしい書き方**
```python
# 単一サンプル向けの関数を書く (シンプル)
def per_sample_fn(x): ...

# あとは合成するだけ
batched = jax.vmap(per_sample_fn)        # バッチ化
fast    = jax.jit(batched)               # 高速化
grad    = jax.grad(batched)              # 微分
```

この **「単一サンプル関数 + 変換の合成」** が JAX の真髄です。

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [02_autodiff.ipynb](02_autodiff.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [章 TOP](../README.md) |